# Pipeline 2 - Breakfast Actions Playbook

Notebook operativo para ejecutar y validar el flujo frame-level de Breakfast Actions en `pipeline/pipeline2_tda_dl`.

## 1) Configuracion

Ajusta rutas y parametros antes de ejecutar las celdas de pipeline.

In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path.cwd()
VIDEOS_DIR = PROJECT_ROOT / 'data' / 'breakfast' / 'videos'
ANNOTATIONS_DIR = PROJECT_ROOT / 'data' / 'breakfast' / 'annotations'
SPLIT_FILE = PROJECT_ROOT / 'data' / 'breakfast' / 'splits.json'
ARTIFACTS_DIR = PROJECT_ROOT / 'pipeline' / 'pipeline2_tda_dl' / 'artifacts_breakfast'

WINDOW_SIZE = 31
STRIDE_TRAIN = 5
STRIDE_VAL = 5
STRIDE_TEST = 1
EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-3
SEED = 1337

print('PROJECT_ROOT:', PROJECT_ROOT)
print('VIDEOS_DIR:', VIDEOS_DIR)
print('ANNOTATIONS_DIR:', ANNOTATIONS_DIR)
print('SPLIT_FILE:', SPLIT_FILE)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)

In [ ]:
def run(cmd: str):
    print('\n$ ' + cmd)
    return subprocess.run(cmd, shell=True, cwd=PROJECT_ROOT, check=True)

## 2) Validaciones rapidas

In [ ]:
assert VIDEOS_DIR.exists(), f'No existe videos_dir: {VIDEOS_DIR}'
assert ANNOTATIONS_DIR.exists(), f'No existe annotations_dir: {ANNOTATIONS_DIR}'
assert SPLIT_FILE.exists(), f'No existe split_file: {SPLIT_FILE}'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('OK: rutas base disponibles')

## 3) Pipeline completo (Breakfast Actions)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.breakfast_manifest_builder ' +
    f'--videos_dir "{VIDEOS_DIR}" ' +
    f'--annotations_dir "{ANNOTATIONS_DIR}" ' +
    f'--split_file "{SPLIT_FILE}" ' +
    f'--output_manifest "{ARTIFACTS_DIR / "manifest.json"}"'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.breakfast_cubical_preprocessing ' +
    f'--dataset_manifest "{ARTIFACTS_DIR / "manifest.json"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "outputs_cubical"}" ' +
    '--sample_fps 3.0'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.breakfast_curves ' +
    f'--input_dir "{ARTIFACTS_DIR / "outputs_cubical"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "outputs_curves"}" ' +
    '--smooth_window 5'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.build_frame_labels ' +
    f'--dataset_manifest "{ARTIFACTS_DIR / "manifest.json"}" ' +
    f'--cubical_manifest "{ARTIFACTS_DIR / "outputs_cubical" / "manifest_cubical.json"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "frame_labels"}" ' +
    '--train_split_names train'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.breakfast_temporal_windows ' +
    f'--cubical_manifest "{ARTIFACTS_DIR / "outputs_cubical" / "manifest_cubical.json"}" ' +
    f'--curves_manifest "{ARTIFACTS_DIR / "outputs_curves" / "manifest_curves.json"}" ' +
    f'--frame_labels_manifest "{ARTIFACTS_DIR / "frame_labels" / "manifest_frame_labels.json"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "windows"}" ' +
    f'--window_size {WINDOW_SIZE} --stride_train {STRIDE_TRAIN} --stride_val {STRIDE_VAL} --stride_test {STRIDE_TEST}'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.train_breakfast_temporal_segmenter ' +
    f'--windows_dir "{ARTIFACTS_DIR / "windows"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "temporal_model"}" ' +
    f'--epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {SEED} ' +
    '--ignore_unknown --class_weighting'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.infer_breakfast_temporal_segmenter ' +
    f'--windows_npz "{ARTIFACTS_DIR / "windows" / "test_windows.npz"}" ' +
    f'--model_checkpoint "{ARTIFACTS_DIR / "temporal_model" / "breakfast_temporal_best.pt"}" ' +
    f'--frame_labels_manifest "{ARTIFACTS_DIR / "frame_labels" / "manifest_frame_labels.json"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "inference"}" ' +
    f'--seed {SEED}'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.decode_breakfast_predictions ' +
    f'--raw_manifest "{ARTIFACTS_DIR / "inference" / "raw_predictions_manifest.json"}" ' +
    f'--output_dir "{ARTIFACTS_DIR / "decoded"}" ' +
    '--kernel_size 5 --min_segment_sec 0.5'
)

In [ ]:
run(
    'python -m pipeline.pipeline2_tda_dl.eval_breakfast_segmentation ' +
    f'--decoded_manifest "{ARTIFACTS_DIR / "decoded" / "decoded_manifest.json"}" ' +
    f'--frame_labels_manifest "{ARTIFACTS_DIR / "frame_labels" / "manifest_frame_labels.json"}" ' +
    '--splits test ' +
    f'--output_json "{ARTIFACTS_DIR / "eval" / "eval_test.json"}"'
)

## 4) Smoke test rapido

Si quieres validar wiring en pocos minutos, limita videos por split y usa 1 epoca.

In [ ]:
SMOKE_DIR = PROJECT_ROOT / 'pipeline' / 'pipeline2_tda_dl' / 'artifacts_breakfast_smoke'
run(
    'python -m pipeline.pipeline2_tda_dl.breakfast_manifest_builder ' +
    f'--videos_dir "{VIDEOS_DIR}" ' +
    f'--annotations_dir "{ANNOTATIONS_DIR}" ' +
    f'--split_file "{SPLIT_FILE}" ' +
    '--max_videos_per_split 2 ' +
    f'--output_manifest "{SMOKE_DIR / "manifest.json"}"'
)